# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
import os
import requests
import json
import ollama
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

In [2]:
# constants
MODEL_GPT = 'gpt-4o-mini'
MODEL_OLLAMA = 'gemma3:4b-it-qat'

In [3]:
class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):

        headers = {
            "User-Agent":
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
        }

        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [4]:
def get_links_user_prompt(website):
    user_prompt = f"以下の ##リンク一覧 は次のウェブサイト内に配置されたURLです。ウェブサイト: {website.url}"
    user_prompt += "与えられえたウェブサイトの会社のパンフレットを作成するのに有用なリンクを選別してください。完全なhttp URLを含むJSONで返答してください。"

    user_prompt += "## リンク一覧:\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [5]:
def get_links(url):
    system_prompt = "あなたはあるウェブサイトに配置されたURLのリストを受け取ります。\
    与えられたウェブサイトの会社のパンフレットを作成するために、必要だと判断したURLを選別してください。 \
    例えば会社概要ページや、採用情報ページの情報を含めてください。\n"

    system_prompt += "以下のような、JSON形式で、選別したURLのリストを返答してください。受け取る際にエラーとなるため、シンプルにJSONのみを含め、説明文などは必ず省略してください。:\n"
    system_prompt += """
    {
        "links": [
            {"type": "about page", "url": "https://full.url/goes/here/about"},
            {"type": "careers page", "url": "https://another.full.url/careers"}
        ]
    }
    """
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[{
            "role": "system",
            "content": system_prompt
        }, {
            "role": "user",
            "content": get_links_user_prompt(website)
        }],
        response_format={"type": "json_object"})
    result = response.choices[0].message.content
    return json.loads(result)

In [6]:
def get_all_details(url):
    result = "ランディングページ:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [7]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"あなたは次のサイトのウェブサイト情報を拝見しています: {company_name}\n"
    user_prompt += f"以下のランディングページと関連ウェブサイトの情報を参照し、会社のパンフレットを作成してください。マークダウン形式で回答してください。.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [8]:
def create_brochure_gpt(company_name, url):
    system_prompt = """
    ## あなたの役割 
    あなたは優秀なコンサルタントで、資料作成のスキルに優れています。
    ## あなたの目的 
    あなたは会社のウェブサイトのランディングページと、関連ウェブサイトの情報元にパンフレットの作成を支援することが目的です。
    ## 必ず守るべきルール
    - マークダウン形式で回答してください。
    - 会社の文化、顧客、採用情報に関する情報を含めてください。
    """

    response = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [9]:
def create_brochure_ollama(company_name, url):
    system_prompt = """
    ## あなたの役割 
    あなたは優秀なコンサルタントで、資料作成のスキルに優れています。
    ## あなたの目的 
    あなたは会社のウェブサイトのランディングページと、関連ウェブサイトの情報元にパンフレットの作成を支援することが目的です。
    ## 必ず守るべきルール
    - マークダウン形式で回答してください。
    - 会社の文化、顧客、採用情報に関する情報を含めてください。
    """

    response = ollama.chat(model=MODEL_OLLAMA,
                           messages=[{
                               "role": "system",
                               "content": system_prompt
                           }, {
                               "role":
                               "user",
                               "content":
                               get_brochure_user_prompt(company_name, url)
                           }])
    display(Markdown(response['message']['content']))

In [10]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
openai = OpenAI()

In [21]:
get_links("https://www.ibm.com/jp-ja")

{'links': [{'type': 'about page',
   'url': 'https://www.ibm.com/jp-ja/about?lnk=hpii1jp'},
  {'type': 'careers page',
   'url': 'https://www.ibm.com/jp-ja/careers?lnk=hpii5jp'},
  {'type': 'history page',
   'url': 'https://www.ibm.com/jp-ja/history?lnk=hpii2jp'}]}

In [ ]:
get_all_details("https://www.ibm.com/jp-ja")

In [20]:
create_brochure_gpt( company_name="日本IBM", url= "https://www.ibm.com/jp-ja")

Found links: {'links': [{'type': 'about page', 'url': 'https://www.ibm.com/jp-ja/about?lnk=hpii1jp'}, {'type': 'careers page', 'url': 'https://www.ibm.com/jp-ja/careers?lnk=hpii5jp'}, {'type': 'history page', 'url': 'https://www.ibm.com/jp-ja/history?lnk=hpii2jp'}, {'type': 'case studies page', 'url': 'https://www.ibm.com/case-studies/jp-ja/?lnk=hprc4jp'}, {'type': 'consulting page', 'url': 'https://www.ibm.com/jp-ja/consulting?lnk=hpfp3jp'}]}


```markdown
# IBM - 日本IBM パンフレット

## 会社概要
- **設立年月日**: 1937年(昭和12年)6月17日
- **所在地**: 東京都港区虎ノ門二丁目6番1号 虎ノ門ヒルズ ステーションタワー
- **代表者**: 山口 明夫 (代表取締役社長執行役員)
- **資本金**: 1,053億円
- **株主**: IBM Japan Holdings合同会社(100%)
- **事業内容**: 情報システムに関わる製品・サービスの提供

## 会社文化
IBMは、イノベーションをもって社会に貢献することを使命としており、企業倫理、環境への配慮、信頼性の高いテクノロジーを通じて地域社会への貢献を目指しています。常に進化を追求し、競争力のあるビジネス環境を構築するためのリーダーシップを発揮しています。

## 最新の取り組み
- **AIと自動化**: 最新のAI技術を用いた「病院業務支援AIソリューション」の提供開始により、医療従事者の業務変革を促進。
- **国際的な連携**: ドイツと日本の宇宙機関間のロボット連携における成果。
- **エンタープライズIT**: IBM Power11の発表により、ITの水準を引き上げる取り組み。

## 注目のコンテンツ
- **BIリーダーシップ**: 経営層向けに提供するビジネスの最新動向をキャッチアップできるレポート「CEOスタディ2025」。
- **AIエージェント**: 人事向けAIエージェントの導入事例を通じて、業務の効率化を支援。

## 採用情報
- **IBMで働く**: IBMでは、スキルを磨くための教育プログラムを用意しており、学びながら成長できる環境が整っています。興味のある方は、[IBMのキャリアページ](https://www.ibm.com/employment)をご覧ください。

## お問い合わせ
- **プレスルーム**: 最新のプレスリリースやニュースを確認できます。詳細は[こちら](https://www.ibm.com/press)。

### ビジョン
「会社はただの組織ではなく、持続可能な未来を築くための触媒である」という理念をもとに、IBMは顧客とテクノロジーの新しい融合を具現化し、変革を加速させます。

### 連絡先
- **住所**: 〒105-5531 東京都港区虎ノ門二丁目6番1号 虎ノ門ヒルズ ステーションタワー
- **ウェブサイト**: [IBM - 日本](https://www.ibm.com/jp)

---
```

このパンフレットは日本IBMの基本情報、会社の文化、最新の取り組み、採用情報をまとめたものです。内容を適宜変更し、必要に応じてデザインを追加することで、印刷用やデジタル配布用の資料として活用できます。

In [12]:
# Get Llama 3.2 to answer
create_brochure_ollama( company_name="日本IBM", url= "https://www.ibm.com/jp-ja")

Found links: {'links': [{'type': 'about page', 'url': 'https://www.ibm.com/jp-ja/about?lnk=hpii1jp'}, {'type': 'history page', 'url': 'https://www.ibm.com/jp-ja/history?lnk=hpii2jp'}, {'type': 'careers page', 'url': 'https://www.ibm.com/jp-ja/careers?lnk=hpii5jp'}]}


## 日本IBM パンフレット

**表紙:**

(日本IBMのロゴと、印象的なAI技術のイメージ画像を使用)

**タイトル:** 未来を創造するパートナー – 日本IBM

**キャッチフレーズ:** ビーン・カタリスト。ビジネスを、社会を、そして未来を変える。

**目次:**

1.  **はじめに** (日本IBMについて)
2.  **日本IBMの強み**
3.  **提供するソリューション**
4.  **成功事例**
5.  **日本IBMの企業文化と価値観**
6.  **お問い合わせ**

---

**1. はじめに - 日本IBMについて**

1937年創業の日本IBMは、日本におけるITソリューションのパイオニアです。IBMグループの一員として、お客様のビジネスの成長を支援し、社会課題の解決に貢献しています。長年にわたる実績と、グローバルなネットワークを活かし、日本のお客様に最適なソリューションを提供します。

*   **会社概要:**
    *   社名: 日本IBM
    *   設立年月日: 1937年6月17日
    *   本社所在地: 〒105-5531 東京都港区虎ノ門二丁目6番1号 虎ノ門ヒルズ ステーションタワー
    *   代表取締役社長執行役員: 山口 明夫
    *   資本金: 1,053億円
    *   従業員数: 30,000名以上 (170カ国以上にわたる従業員)
*   **日本IBMのミッション:** お客様のビジネスの成長と、社会課題の解決に貢献する。
*   **日本IBMの強み:**
    *   グローバルなネットワークと最先端技術
    *   長年の実績とお客様との信頼関係
    *   高度なコンサルティング力
    *   多様なソリューションポートフォリオ
    *   地域に根差した事業展開

---

**2. 日本IBMの強み**

日本IBMは、以下の強みを通じてお客様のビジネスを支援します。

*   **AIソリューション:**
    *   IBM Watson x.ai: ビジネス向けに設計されたAIを活用し、業務効率化、意思決定の迅速化、顧客体験の向上を実現します。
    *   watsonx Orchestrate: AIモデルを統合、管理、実行するためのプラットフォームを提供します。
    *   IBM Guardium Data Security Center: データの保護を強化し、コンプライアンス要件を満たすためのセキュリティソリューションを提供します。
*   **クラウドソリューション:**
    *   IBM Cloud: 安全で信頼性の高いクラウドプラットフォームを提供し、お客様のデジタル変革を支援します。
    *   Red Hat OpenShift: コンテナ化されたアプリケーションを効率的に実行するためのプラットフォームを提供します。
*   **コンサルティング:**
    *   IBM Consulting: 業界に特化した専門知識と経験に基づいて、お客様のビジネス戦略、IT戦略、デジタル変革を支援します。
    *   幅広い業界知識 (金融、製造、医療、小売など)
*   **グローバルネットワーク:** IBMのグローバルなリソースと専門知識を活用し、お客様を世界規模でサポートします。

---

**3. 提供するソリューション**

日本IBMは、お客様のビジネス課題を解決するための、幅広いソリューションを提供します。

*   **AIソリューション:**
    *   データ分析・予測
    *   顧客体験の向上
    *   業務自動化
    *   セキュリティ強化
*   **クラウドソリューション:**
    *   データ分析
    *   アプリケーション開発
    *   セキュリティ
*   **コンサルティング:**
    *   ビジネス戦略
    *   IT戦略
    *   デジタル変革
    *   セキュリティ
*   **技術サポート:** 専門知識を持つ技術者が、お客様のソリューションを導入、運用、保守します。

---

**4. 成功事例**

(以下は具体的な成功事例の数例。貴社の実績に合わせて追記・修正してください。)

*   **株式会社かんぽ生命保険:** サービス障害の検知精度を向上、原因調査時間を大幅に短縮。IBM SREチームのオブザーバビリティー導入事例。
*   **株式会社みずほ銀行:** 問題調査に要する時間を短縮。ITオートメーションソリューション導入による信頼性向上。
*   **日本電気株式会社（NEC）:** クラウド支出を削減。IBM Cloudとの連携によるコスト最適化。
*   **Winblonゲートボール協会:** AIを活用してファン・エンゲージメントを高める新たなAI機能を発表。

---

**5. 日本IBMの企業文化と価値観**

日本IBMは、以下の企業文化と価値観を大切にしています。

*   **イノベーション:** 新しいアイデアと技術を積極的に取り入れ、社会に貢献する。
*   **多様性:** 多様な人材が活躍できる環境づくりに努める。
*   **顧客第一:** お客様のニーズを理解し、最適なソリューションを提供すること。
*   **社会貢献:** 地域社会との連携を強化し、社会課題の解決に貢献する。

---

**6. お問い合わせ**

*   **本社:** 〒105-5531 東京都港区虎ノ門二丁目6番1号 虎ノ門ヒルズ ステーションタワー
*   **電話番号:** (貴社の連絡先を記載)
*   **ウェブサイト:** [貴社のウェブサイトアドレス]

(日本IBMの企業文化、沿革、歴史に関する情報を網羅的に記載するスペースがあれば追記してください。IBM Labにおける研究開発への取り組みなども紹介すると、より魅力的なパンフレットになります。)

**(バックページ)**

IBMおよび日本IBMについて | IBM (URL)

**注:** 上記は、提供する情報と想定される内容のテンプレートです。貴社の具体的な情報に合わせて内容を追記・修正してください。特に、実績事例は、貴社の強みと実績をアピールする重要な要素です。
